In [1]:
!nvidia-smi | head -10
import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}  '
      f'device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

Sun May 17 14:52:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   46C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
torch=2.10.0+cu128  cuda=True  device=NVIDIA L4


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'
%cd $REPO_DIR

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


In [3]:
# 1. 与真实 train_sac 默认完全一致的 baseline
!python -m scripts.profile_train \
    --flow wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy \
    --num-envs 6 --vector-mode async --device cuda \
    --updates-per-step 4 \
    --output-json experiments/profile/l4_n6_async_cuda_utd4.json

[profile] device=cuda  num_envs=6 (async)  obs_dim=40  action_dim=2  priv_dim=0  target=4500 transitions
[profile] warmup complete at 1500 transitions; measuring...
  device=cuda  num_envs=6 (async)  UTD=4  batch=256  asym=False  ln=False
  probe=s0  history=4  geom=upstream  tgt=1.5  obj=efficiency_v2
----------------------------------------------------------------------------------------
  measured 3000 transitions in 35.24s  ->  85.1 transitions/s   14.2 env.step/s   56.8 updates/s
----------------------------------------------------------------------------------------
  bucket             count    total_ms   mean_ms    p50_ms    p95_ms   % wallclock
  agent_act            500       495.0     0.990     0.981     1.049          1.4%
  env_step             500     10891.5    21.783    18.641    21.647         30.9%
  replay_add           500        37.5     0.075     0.071     0.092          0.1%
  replay_sample       2000       938.9     0.469     0.463     0.532          2.7%
  sac_

In [4]:
# 2. 加上 thesis 主线的 LayerNorm + AsymCritic(因为它们影响 update 端开销)
!python -m scripts.profile_train \
    --flow wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy \
    --num-envs 6 --vector-mode async --device cuda \
    --updates-per-step 4 \
    --use-layernorm --use-asymmetric-critic \
    --output-json experiments/profile/l4_n6_async_cuda_utd4_thesis.json

[profile] device=cuda  num_envs=6 (async)  obs_dim=40  action_dim=2  priv_dim=2  target=4500 transitions
[profile] warmup complete at 1500 transitions; measuring...
  device=cuda  num_envs=6 (async)  UTD=4  batch=256  asym=True  ln=True
  probe=s0  history=4  geom=upstream  tgt=1.5  obj=efficiency_v2
----------------------------------------------------------------------------------------
  measured 3000 transitions in 36.87s  ->  81.4 transitions/s   13.6 env.step/s   54.2 updates/s
----------------------------------------------------------------------------------------
  bucket             count    total_ms   mean_ms    p50_ms    p95_ms   % wallclock
  agent_act            500       543.1     1.086     1.080     1.156          1.5%
  env_step             500      8160.5    16.321    17.264    20.912         22.1%
  replay_add           500        40.8     0.082     0.077     0.098          0.1%
  replay_sample       2000      1119.0     0.560     0.554     0.619          3.0%
  sac_up

In [5]:
%%bash
# 3. num_envs scaling — 看 env_step 是否随 num_envs 线性 scale
for n in 1 4 6 8 12; do
  python -m scripts.profile_train \
    --flow wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy \
    --num-envs $n --vector-mode async --device cuda \
    --updates-per-step 4 \
    --output-json experiments/profile/l4_n${n}_async_cuda_utd4.json
done


[profile] device=cuda  num_envs=1 (async)  obs_dim=40  action_dim=2  priv_dim=0  target=4500 transitions
[profile] warmup complete at 1500 transitions; measuring...
  device=cuda  num_envs=1 (single)  UTD=4  batch=256  asym=False  ln=False
  probe=s0  history=4  geom=upstream  tgt=1.5  obj=efficiency_v2
----------------------------------------------------------------------------------------
  measured 3000 transitions in 178.37s  ->  16.8 transitions/s   16.8 env.step/s   67.3 updates/s
----------------------------------------------------------------------------------------
  bucket             count    total_ms   mean_ms    p50_ms    p95_ms   % wallclock
  agent_act           3000      3238.4     1.079     1.073     1.145          1.8%
  env_step            3000     31262.3    10.421    10.377    11.005         17.5%
  replay_add          3000        90.5     0.030     0.028     0.044          0.1%
  replay_sample      12000      5601.7     0.467     0.465     0.506          3.1%
  sa